# CS5224 Data Curation — Stage 1C

## Safe field normalization

Stage 1B established provenance for all **177,638** source records.

Stage 1C adds normalized versions of the fields that can be cleaned safely without interpreting product identity:

- `source_category_normalized`
- `product_name_normalized`
- `brand_normalized`
- `product_price`
- `original_price`
- `in_stock`
- `has_offer`

The corresponding raw fields are retained unchanged.

### Stage 1C deliberately does NOT

- match products across retailers;
- create canonical product IDs;
- map products into the final eight CartOptix categories;
- infer missing brands;
- remove product variants or descriptive words;
- parse pack sizes/specifications;
- calculate promotional or multibuy prices;
- remove duplicates.

Specification/pack parsing is deferred to Stage 1D.


## 1. Imports and project paths

In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd

# Notebook location:
# CartOptix/CartOptix/scripts

PROJECT_ROOT = Path.cwd().parent
STAGING_DATA_DIR = PROJECT_ROOT / "data" / "staging"

STAGE1B_FILE = STAGING_DATA_DIR / "all_retailers_stage1b_raw.csv"

print("Current notebook folder:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nStaging data directory:")
print(STAGING_DATA_DIR)

print("\nStage 1B input:")
print(STAGE1B_FILE)

print("\nExists:", STAGE1B_FILE.exists())

assert STAGE1B_FILE.exists(), (
    "Stage 1B checkpoint not found. "
    "Run Data_Curation_Stage1B.ipynb first."
)


Current notebook folder:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\scripts

Project root:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix

Staging data directory:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging

Stage 1B input:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\all_retailers_stage1b_raw.csv

Exists: True


## 2. Define the expected Stage 1B schema

In [2]:
STAGE1B_COLUMNS = [
    "retailer",
    "source_file",
    "source_sheet",
    "source_row_number",
    "source_record_id",
    "source_category_raw",
    "product_name_raw",
    "product_price_raw",
    "original_price_raw",
    "specification_raw",
    "savings_raw",
    "stock_raw",
    "offer_raw",
    "image_url_raw",
    "product_url_raw",
    "comments_raw",
    "brand_raw",
]

EXPECTED_TOTAL_ROWS = 177_638
EXPECTED_RETAILER_COUNTS = {
    "redmart": 97_779,
    "fairprice": 69_402,
    "coldstorage": 10_457,
}


## 3. Load the Stage 1B checkpoint

In [3]:
stage1b = pd.read_csv(
    STAGE1B_FILE,
    dtype={
        "retailer": "string",
        "source_file": "string",
        "source_sheet": "string",
        "source_record_id": "string",
    }
)

print("Loaded Stage 1B shape:")
print(stage1b.shape)

display(stage1b.head(3))


Loaded Stage 1B shape:
(177638, 17)


,retailer,source_file,source_sheet,source_row_number,source_record_id,source_category_raw,product_name_raw,product_price_raw,original_price_raw,specification_raw,savings_raw,stock_raw,offer_raw,image_url_raw,product_url_raw,comments_raw,brand_raw
0,redmart,redmart_all_products_final.xlsx,Sheet1,2,redmart_00000002,shop-chilled-desserts,Sojasun Natural Soya Yoghurt,12.0,NaN,6 × 100 g,NaN,Yes,No,https://sg-live-01.slatic.net/p/5eaa6087b6c250...,https://www.lazada.sg/products/pdp-i303336401-...,NaN,Sojasun
1,redmart,redmart_all_products_final.xlsx,Sheet1,3,redmart_00000003,shop-chilled-desserts,Monoprix Chocolate Custard Dessert Cream 4x125G,10.9,NaN,4 × 125 g,Spend $45 + Free Gift,Yes,Yes,https://sg-test-11.slatic.net/p/9f86cfbce0b472...,https://www.lazada.sg/products/pdp-i462726209-...,NaN,Monoprix
2,redmart,redmart_all_products_final.xlsx,Sheet1,4,redmart_00000004,shop-chilled-desserts,Top Gourmet Cheng Tng,3.0,NaN,400 g,NaN,Yes,No,https://sg-live-01.slatic.net/p/7579bbca2beaa0...,https://www.lazada.sg/products/pdp-i303250731-...,NaN,Top Gourmet


## 4. Validate the Stage 1B input before normalization

In [4]:
actual_columns = stage1b.columns.tolist()

assert actual_columns == STAGE1B_COLUMNS, (
    "Stage 1B columns do not match the expected schema.\n"
    f"Expected: {STAGE1B_COLUMNS}\n"
    f"Actual:   {actual_columns}"
)

assert len(stage1b) == EXPECTED_TOTAL_ROWS, (
    f"Expected {EXPECTED_TOTAL_ROWS:,} rows, "
    f"but loaded {len(stage1b):,}."
)

assert stage1b["source_record_id"].duplicated().sum() == 0, (
    "Duplicate source_record_id values found."
)

assert stage1b["source_record_id"].isna().sum() == 0, (
    "Missing source_record_id values found."
)

for retailer, expected_count in EXPECTED_RETAILER_COUNTS.items():
    actual_count = (stage1b["retailer"] == retailer).sum()

    assert actual_count == expected_count, (
        f"{retailer}: expected {expected_count:,}, "
        f"found {actual_count:,}."
    )

print("PASS: Stage 1B input validation succeeded.")


PASS: Stage 1B input validation succeeded.


## 5. Preserve a raw-field snapshot

This snapshot is used later to verify that Stage 1C has not overwritten any raw source values.


In [5]:
RAW_FIELDS = [
    "source_category_raw",
    "product_name_raw",
    "product_price_raw",
    "original_price_raw",
    "specification_raw",
    "savings_raw",
    "stock_raw",
    "offer_raw",
    "image_url_raw",
    "product_url_raw",
    "comments_raw",
    "brand_raw",
]

raw_snapshot = stage1b[RAW_FIELDS].copy(deep=True)

print(
    f"Raw snapshot created: "
    f"{raw_snapshot.shape[0]:,} rows x "
    f"{raw_snapshot.shape[1]} columns"
)


Raw snapshot created: 177,638 rows x 12 columns


## 6. Define conservative text normalization

The function below:

1. preserves null values;
2. applies Unicode NFKC normalization;
3. trims leading/trailing whitespace;
4. collapses repeated whitespace;
5. converts text to case-insensitive form using `casefold()`.

It does **not** remove punctuation, brand terms, variant words, sizes, words such as `zero`, `organic`, `premium`, etc.

Those details may be important for exact product matching later.


In [6]:
def normalize_text(value):
    """
    Conservatively normalize text while retaining product meaning.

    Returns pd.NA for missing/blank input.
    """

    if pd.isna(value):
        return pd.NA

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    # Collapse tabs/newlines/multiple spaces into one space.
    text = re.sub(r"\s+", " ", text).strip()

    if text == "":
        return pd.NA

    return text.casefold()


## 7. Define brand normalization

Brand normalization uses the same conservative text rules.

The source value `*`, observed in the Cold Storage export, is treated as a placeholder rather than a verified brand. The original value remains in `brand_raw`; only `brand_normalized` becomes null.


In [7]:
def normalize_brand(value):
    """
    Normalize a brand without guessing a missing brand.
    """

    text = normalize_text(value)

    if pd.isna(text):
        return pd.NA

    # Known placeholder in the supplied Cold Storage data.
    if text == "*":
        return pd.NA

    return text


## 8. Define price normalization

Prices are normalized into numeric values with two decimal places.

Examples:

- `12.63` → `12.63`
- `"$12.63 "` → `12.63`
- `"$1,234.50"` → `1234.50`
- missing source value → null

No missing price is invented. No promotion is applied here.


In [8]:
def normalize_price(value):
    """
    Convert a raw price into a numeric two-decimal value.

    Returns pd.NA when the source value is missing or cannot
    be parsed safely.
    """

    if pd.isna(value):
        return pd.NA

    # Numeric values already read by pandas.
    if isinstance(
        value,
        (int, float, np.integer, np.floating)
    ) and not isinstance(value, (bool, np.bool_)):

        if not np.isfinite(float(value)):
            return pd.NA

        return round(float(value), 2)

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    ).strip()

    text = (
        text
        .replace("$", "")
        .replace(",", "")
    )

    text = re.sub(r"\s+", "", text)

    if text == "":
        return pd.NA

    try:
        parsed = float(text)
    except ValueError:
        return pd.NA

    if not np.isfinite(parsed):
        return pd.NA

    return round(parsed, 2)


## 9. Define Yes/No normalization

`Yes` becomes `True`, and `No` becomes `False`.

Any other value is left as null so that an unexpected value is visible rather than silently interpreted.


In [9]:
def normalize_yes_no(value):
    """
    Convert source Yes/No values into nullable booleans.
    """

    if pd.isna(value):
        return pd.NA

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    ).strip().casefold()

    if text == "yes":
        return True

    if text == "no":
        return False

    return pd.NA


## 10. Apply Stage 1C normalization

In [10]:
stage1c = stage1b.copy()

# Text fields
stage1c["source_category_normalized"] = (
    stage1c["source_category_raw"]
    .apply(normalize_text)
    .astype("string")
)

stage1c["product_name_normalized"] = (
    stage1c["product_name_raw"]
    .apply(normalize_text)
    .astype("string")
)

stage1c["brand_normalized"] = (
    stage1c["brand_raw"]
    .apply(normalize_brand)
    .astype("string")
)

# Price fields
stage1c["product_price"] = (
    stage1c["product_price_raw"]
    .apply(normalize_price)
    .astype("Float64")
)

stage1c["original_price"] = (
    stage1c["original_price_raw"]
    .apply(normalize_price)
    .astype("Float64")
)

# Boolean fields
stage1c["in_stock"] = (
    stage1c["stock_raw"]
    .apply(normalize_yes_no)
    .astype("boolean")
)

stage1c["has_offer"] = (
    stage1c["offer_raw"]
    .apply(normalize_yes_no)
    .astype("boolean")
)

print("Stage 1C shape:")
print(stage1c.shape)


Stage 1C shape:
(177638, 24)


The Stage 1C table should contain **24 columns**:

- 17 Stage 1B columns
- 7 newly normalized columns


## 11. Put Stage 1C columns into a clear order

In [11]:
STAGE1C_COLUMNS = [
    # Provenance
    "retailer",
    "source_file",
    "source_sheet",
    "source_row_number",
    "source_record_id",

    # Category
    "source_category_raw",
    "source_category_normalized",

    # Product name
    "product_name_raw",
    "product_name_normalized",

    # Brand
    "brand_raw",
    "brand_normalized",

    # Specification remains raw until Stage 1D
    "specification_raw",

    # Prices
    "product_price_raw",
    "product_price",
    "original_price_raw",
    "original_price",

    # Promotion text remains raw
    "savings_raw",

    # Stock
    "stock_raw",
    "in_stock",

    # Offer flag
    "offer_raw",
    "has_offer",

    # URLs/comments remain raw
    "image_url_raw",
    "product_url_raw",
    "comments_raw",
]

stage1c = stage1c[STAGE1C_COLUMNS]

print("Stage 1C columns:")
print("-" * 50)

for i, column in enumerate(stage1c.columns, start=1):
    print(f"{i:2d}. {column}")

assert len(stage1c.columns) == 24


Stage 1C columns:
--------------------------------------------------
 1. retailer
 2. source_file
 3. source_sheet
 4. source_row_number
 5. source_record_id
 6. source_category_raw
 7. source_category_normalized
 8. product_name_raw
 9. product_name_normalized
10. brand_raw
11. brand_normalized
12. specification_raw
13. product_price_raw
14. product_price
15. original_price_raw
16. original_price
17. savings_raw
18. stock_raw
19. in_stock
20. offer_raw
21. has_offer
22. image_url_raw
23. product_url_raw
24. comments_raw


## 12. Verify that raw source fields were not changed

In [12]:
assert stage1c[RAW_FIELDS].equals(raw_snapshot), (
    "At least one raw source field changed during Stage 1C."
)

print("PASS: All raw source fields remain unchanged.")


PASS: All raw source fields remain unchanged.


## 13. Validate normalized text fields

In [13]:
missing_categories = (
    stage1c["source_category_normalized"]
    .isna()
    .sum()
)

missing_product_names = (
    stage1c["product_name_normalized"]
    .isna()
    .sum()
)

missing_brands = (
    stage1c["brand_normalized"]
    .isna()
    .sum()
)

print(
    "Missing normalized categories:",
    f"{missing_categories:,}"
)

print(
    "Missing normalized product names:",
    f"{missing_product_names:,}"
)

print(
    "Missing normalized brands:",
    f"{missing_brands:,}"
)

# Product name and source category are required for this source set.
assert missing_categories == 0
assert missing_product_names == 0

# Missing brands are allowed and are reported, not invented.

print("\nPASS: Required normalized text fields are present.")


Missing normalized categories: 0
Missing normalized product names: 0
Missing normalized brands: 6,411

PASS: Required normalized text fields are present.


## 14. Inspect raw versus normalized text examples

In [14]:
display(
    stage1c[
        [
            "retailer",
            "source_record_id",
            "source_category_raw",
            "source_category_normalized",
            "product_name_raw",
            "product_name_normalized",
            "brand_raw",
            "brand_normalized",
        ]
    ].head(15)
)


,retailer,source_record_id,source_category_raw,source_category_normalized,product_name_raw,product_name_normalized,brand_raw,brand_normalized
0,redmart,redmart_00000002,shop-chilled-desserts,shop-chilled-desserts,Sojasun Natural Soya Yoghurt,sojasun natural soya yoghurt,Sojasun,sojasun
1,redmart,redmart_00000003,shop-chilled-desserts,shop-chilled-desserts,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix chocolate custard dessert cream 4x125g,Monoprix,monoprix
2,redmart,redmart_00000004,shop-chilled-desserts,shop-chilled-desserts,Top Gourmet Cheng Tng,top gourmet cheng tng,Top Gourmet,top gourmet
3,redmart,redmart_00000005,shop-chilled-desserts,shop-chilled-desserts,La Laitiere Vanilla Rice Pudding Riz Au Lait 4...,la laitiere vanilla rice pudding riz au lait 4...,La Laitiere,la laitiere
4,redmart,redmart_00000006,shop-chilled-desserts,shop-chilled-desserts,Monoprix Flan Caramel Vanilla Pudding 4X100G,monoprix flan caramel vanilla pudding 4x100g,Monoprix,monoprix
5,redmart,redmart_00000007,shop-chilled-desserts,shop-chilled-desserts,Fortune Sweetened Chin Chow 1KG,fortune sweetened chin chow 1kg,Fortune,fortune
6,redmart,redmart_00000008,shop-chilled-desserts,shop-chilled-desserts,RedMart Trio Swiss Roll (Singapore Special),redmart trio swiss roll (singapore special),RedMart,redmart
7,redmart,redmart_00000009,shop-chilled-desserts,shop-chilled-desserts,Arla Protein Chocolate Pudding 200G - Lactose ...,arla protein chocolate pudding 200g - lactose ...,Arla,arla
8,redmart,redmart_00000010,shop-chilled-desserts,shop-chilled-desserts,Fortune Soya Beancurd Almond 360G,fortune soya beancurd almond 360g,Fortune,fortune
9,redmart,redmart_00000011,shop-chilled-desserts,shop-chilled-desserts,Arla Protein Hazelnut Pudding 200G - Lactose Free,arla protein hazelnut pudding 200g - lactose free,Arla,arla


## 15. Report brand placeholders and missing brands

These records are retained. A missing normalized brand is **not** automatically inferred from the product name.


In [15]:
brand_raw_as_string = (
    stage1c["brand_raw"]
    .astype("string")
    .str.strip()
)

star_brand_count = (
    brand_raw_as_string
    .eq("*")
    .fillna(False)
    .sum()
)

print(
    "Raw brand='*' placeholder records:",
    f"{star_brand_count:,}"
)

print("\nMissing normalized brands by retailer:")

print(
    stage1c
    .groupby("retailer")["brand_normalized"]
    .apply(lambda s: s.isna().sum())
)


Raw brand='*' placeholder records: 224

Missing normalized brands by retailer:
retailer
coldstorage     232
fairprice      1063
redmart        5116
Name: brand_normalized, dtype: int64


## 16. Validate product prices

In [16]:
product_price_parse_failures = (
    stage1c["product_price_raw"].notna()
    & stage1c["product_price"].isna()
)

negative_product_prices = (
    stage1c["product_price"].notna()
    & (stage1c["product_price"] < 0)
)

zero_product_prices = (
    stage1c["product_price"] == 0
).fillna(False)

print(
    "Product price parse failures:",
    f"{product_price_parse_failures.sum():,}"
)

print(
    "Negative product prices:",
    f"{negative_product_prices.sum():,}"
)

print(
    "Zero product prices:",
    f"{zero_product_prices.sum():,}"
)

assert product_price_parse_failures.sum() == 0
assert negative_product_prices.sum() == 0

print(
    "\nPASS: All supplied product prices parsed safely. "
    "Zero prices, if any, are retained for later review."
)


Product price parse failures: 0
Negative product prices: 0
Zero product prices: 2

PASS: All supplied product prices parsed safely. Zero prices, if any, are retained for later review.


Zero-priced source records are **not deleted or changed** in Stage 1C. They are reported for later curation because a zero may represent a source-data issue rather than a valid retail price.


In [17]:
if zero_product_prices.sum() > 0:
    display(
        stage1c.loc[
            zero_product_prices,
            [
                "retailer",
                "source_record_id",
                "product_name_raw",
                "product_price_raw",
                "product_price",
                "product_url_raw",
            ]
        ]
    )


,retailer,source_record_id,product_name_raw,product_price_raw,product_price,product_url_raw
110226,fairprice,fairprice_00012449,Meiji Chocolate Block - Black,0.0,0.0,https://www.fairprice.com.sg/product/meiji-cho...
164532,fairprice,fairprice_00066755,Daisho Crab Miso Hot Pot Soup,0.0,0.0,https://www.fairprice.com.sg/product/daisho-cr...


## 17. Validate original prices

In [18]:
original_price_parse_failures = (
    stage1c["original_price_raw"].notna()
    & stage1c["original_price"].isna()
)

negative_original_prices = (
    stage1c["original_price"].notna()
    & (stage1c["original_price"] < 0)
)

print(
    "Original-price values supplied:",
    f"{stage1c['original_price_raw'].notna().sum():,}"
)

print(
    "Original-price values missing:",
    f"{stage1c['original_price_raw'].isna().sum():,}"
)

print(
    "Original-price parse failures:",
    f"{original_price_parse_failures.sum():,}"
)

print(
    "Negative original prices:",
    f"{negative_original_prices.sum():,}"
)

assert original_price_parse_failures.sum() == 0
assert negative_original_prices.sum() == 0

print(
    "\nPASS: Every supplied original price parsed safely. "
    "Missing original prices remain null."
)


Original-price values supplied: 33,551
Original-price values missing: 144,087
Original-price parse failures: 0
Negative original prices: 0

PASS: Every supplied original price parsed safely. Missing original prices remain null.


## 18. Inspect raw versus normalized price examples

In [19]:
display(
    stage1c.loc[
        stage1c["original_price_raw"].notna(),
        [
            "retailer",
            "source_record_id",
            "product_name_raw",
            "product_price_raw",
            "product_price",
            "original_price_raw",
            "original_price",
        ]
    ].head(15)
)


,retailer,source_record_id,product_name_raw,product_price_raw,product_price,original_price_raw,original_price
10,redmart,redmart_00000012,Sainsbury's Tiramisu (Fresh From Italy),15.21,15.21,$16.90,16.9
21,redmart,redmart_00000023,Sainsbury's Creme Brulee 2X90G,9.72,9.72,$10.80,10.8
32,redmart,redmart_00000034,Bonne Maman Creme Brulee 2 X 100G,10.32,10.32,$12.90,12.9
44,redmart,redmart_00000046,Fortune Soya Beancurd Bandung 360G,1.75,1.75,$1.86,1.86
48,redmart,redmart_00000050,Sainsbury's Caramel Panna Cotta Desserts 2X120G,6.32,6.32,$7.90,7.9
62,redmart,redmart_00000064,Nomadic Power Pot Smooth Caramel 190G,6.02,6.02,$8.85,8.85
72,redmart,redmart_00000074,Nomadic Power Pot Chocolate Brownie 190G,7.96,7.96,$8.85,8.85
85,redmart,redmart_00000087,Belle France Vanilla Cream - By Le Petit Depot,8.49,8.49,$10.61,10.61
86,redmart,redmart_00000088,Carrefour Coffee Liegeois - By Le Petit Depot,8.21,8.21,$10.26,10.26
89,redmart,redmart_00000091,Carrefour Madeleine Shells - By Le Petit Depot,12.51,12.51,$15.64,15.64


## 19. Validate stock normalization

In [20]:
unknown_stock_values = (
    stage1c["stock_raw"].notna()
    & stage1c["in_stock"].isna()
)

print(
    "Unknown/unparsed stock values:",
    f"{unknown_stock_values.sum():,}"
)

print("\nNormalized stock counts:")
print(
    stage1c["in_stock"]
    .value_counts(dropna=False)
)

assert unknown_stock_values.sum() == 0

print("\nPASS: All stock values normalized to True/False.")


Unknown/unparsed stock values: 0

Normalized stock counts:
in_stock
True     146553
False     31085
Name: count, dtype: Int64

PASS: All stock values normalized to True/False.


## 20. Validate offer normalization

In [21]:
unknown_offer_values = (
    stage1c["offer_raw"].notna()
    & stage1c["has_offer"].isna()
)

print(
    "Unknown/unparsed offer values:",
    f"{unknown_offer_values.sum():,}"
)

print("\nNormalized offer counts:")
print(
    stage1c["has_offer"]
    .value_counts(dropna=False)
)

assert unknown_offer_values.sum() == 0

print("\nPASS: All offer values normalized to True/False.")


Unknown/unparsed offer values: 0

Normalized offer counts:
has_offer
False    130332
True      47306
Name: count, dtype: Int64

PASS: All offer values normalized to True/False.


## 21. Verify row counts and provenance were preserved

In [22]:
assert len(stage1c) == EXPECTED_TOTAL_ROWS

assert stage1c["source_record_id"].nunique() == EXPECTED_TOTAL_ROWS

assert stage1c["source_record_id"].duplicated().sum() == 0

for retailer, expected_count in EXPECTED_RETAILER_COUNTS.items():
    actual_count = (
        stage1c["retailer"] == retailer
    ).sum()

    assert actual_count == expected_count

print("PASS: Stage 1C preserved all rows and provenance IDs.")


PASS: Stage 1C preserved all rows and provenance IDs.


## 22. Build a Stage 1C normalization report

In [23]:
report_rows = []

for retailer, df in stage1c.groupby(
    "retailer",
    sort=False
):
    raw_brand_string = (
        df["brand_raw"]
        .astype("string")
        .str.strip()
    )

    report_rows.append(
        {
            "retailer": retailer,
            "rows": len(df),
            "missing_normalized_category": int(
                df["source_category_normalized"]
                .isna()
                .sum()
            ),
            "missing_normalized_product_name": int(
                df["product_name_normalized"]
                .isna()
                .sum()
            ),
            "missing_normalized_brand": int(
                df["brand_normalized"]
                .isna()
                .sum()
            ),
            "brand_star_placeholder": int(
                raw_brand_string
                .eq("*")
                .fillna(False)
                .sum()
            ),
            "product_price_parse_failures": int(
                (
                    df["product_price_raw"].notna()
                    & df["product_price"].isna()
                ).sum()
            ),
            "zero_product_prices": int(
                (df["product_price"] == 0)
                .fillna(False)
                .sum()
            ),
            "negative_product_prices": int(
                (
                    df["product_price"].notna()
                    & (df["product_price"] < 0)
                ).sum()
            ),
            "original_price_supplied": int(
                df["original_price_raw"]
                .notna()
                .sum()
            ),
            "original_price_parse_failures": int(
                (
                    df["original_price_raw"].notna()
                    & df["original_price"].isna()
                ).sum()
            ),
            "unknown_stock_values": int(
                (
                    df["stock_raw"].notna()
                    & df["in_stock"].isna()
                ).sum()
            ),
            "unknown_offer_values": int(
                (
                    df["offer_raw"].notna()
                    & df["has_offer"].isna()
                ).sum()
            ),
        }
    )

normalization_report = pd.DataFrame(report_rows)

display(normalization_report)


,retailer,rows,missing_normalized_category,missing_normalized_product_name,missing_normalized_brand,brand_star_placeholder,product_price_parse_failures,zero_product_prices,negative_product_prices,original_price_supplied,original_price_parse_failures,unknown_stock_values,unknown_offer_values
0,redmart,97779,0,0,5116,0,0,0,0,26975,0,0,0
1,fairprice,69402,0,0,1063,0,0,2,0,4623,0,0,0
2,coldstorage,10457,0,0,232,224,0,0,0,1953,0,0,0


## 23. Save retailer-specific Stage 1C checkpoints

In [24]:
for retailer in [
    "redmart",
    "fairprice",
    "coldstorage",
]:
    retailer_df = (
        stage1c.loc[
            stage1c["retailer"] == retailer
        ]
        .copy()
    )

    output_file = (
        STAGING_DATA_DIR
        / f"{retailer}_stage1c_normalized.csv"
    )

    retailer_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8"
    )

    print(
        f"Saved {retailer}: "
        f"{len(retailer_df):,} rows -> "
        f"{output_file}"
    )


Saved redmart: 97,779 rows -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\redmart_stage1c_normalized.csv
Saved fairprice: 69,402 rows -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\fairprice_stage1c_normalized.csv
Saved coldstorage: 10,457 rows -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\coldstorage_stage1c_normalized.csv


## 24. Save the combined Stage 1C checkpoint

In [25]:
STAGE1C_OUTPUT_FILE = (
    STAGING_DATA_DIR
    / "all_retailers_stage1c_normalized.csv"
)

stage1c.to_csv(
    STAGE1C_OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print("Saved combined Stage 1C dataset:")
print(STAGE1C_OUTPUT_FILE)


Saved combined Stage 1C dataset:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\all_retailers_stage1c_normalized.csv


## 25. Save the Stage 1C normalization report

In [26]:
REPORT_OUTPUT_FILE = (
    STAGING_DATA_DIR
    / "stage1c_normalization_report.csv"
)

normalization_report.to_csv(
    REPORT_OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print("Saved Stage 1C normalization report:")
print(REPORT_OUTPUT_FILE)


Saved Stage 1C normalization report:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\stage1c_normalization_report.csv


## 26. Verify all Stage 1C output files exist

In [27]:
OUTPUT_FILES = [
    STAGING_DATA_DIR / "redmart_stage1c_normalized.csv",
    STAGING_DATA_DIR / "fairprice_stage1c_normalized.csv",
    STAGING_DATA_DIR / "coldstorage_stage1c_normalized.csv",
    STAGE1C_OUTPUT_FILE,
    REPORT_OUTPUT_FILE,
]

for file_path in OUTPUT_FILES:
    print(
        f"{file_path.name:42s} -> "
        f"{file_path.exists()}"
    )

    assert file_path.exists()

print("\nPASS: All Stage 1C output files were created.")


redmart_stage1c_normalized.csv             -> True
fairprice_stage1c_normalized.csv           -> True
coldstorage_stage1c_normalized.csv         -> True
all_retailers_stage1c_normalized.csv       -> True
stage1c_normalization_report.csv           -> True

PASS: All Stage 1C output files were created.


## 27. Final Stage 1C validation

In [28]:
print("=" * 72)
print("STAGE 1C FINAL VALIDATION")
print("=" * 72)

print(
    f"Total records              : "
    f"{len(stage1c):,}"
)

print(
    f"Unique source IDs          : "
    f"{stage1c['source_record_id'].nunique():,}"
)

print(
    f"Duplicate source IDs       : "
    f"{stage1c['source_record_id'].duplicated().sum():,}"
)

print(
    f"Number of columns          : "
    f"{len(stage1c.columns)}"
)

print(
    f"Missing product names      : "
    f"{stage1c['product_name_normalized'].isna().sum():,}"
)

print(
    f"Missing source categories  : "
    f"{stage1c['source_category_normalized'].isna().sum():,}"
)

print(
    f"Missing normalized brands  : "
    f"{stage1c['brand_normalized'].isna().sum():,}"
)

print(
    f"Product-price parse errors : "
    f"{product_price_parse_failures.sum():,}"
)

print(
    f"Original-price parse errors: "
    f"{original_price_parse_failures.sum():,}"
)

print(
    f"Unknown stock values       : "
    f"{unknown_stock_values.sum():,}"
)

print(
    f"Unknown offer values       : "
    f"{unknown_offer_values.sum():,}"
)

print("=" * 72)

assert len(stage1c) == 177_638
assert stage1c["source_record_id"].nunique() == 177_638
assert stage1c["source_record_id"].duplicated().sum() == 0
assert len(stage1c.columns) == 24

assert stage1c["product_name_normalized"].isna().sum() == 0
assert stage1c["source_category_normalized"].isna().sum() == 0

assert product_price_parse_failures.sum() == 0
assert original_price_parse_failures.sum() == 0
assert unknown_stock_values.sum() == 0
assert unknown_offer_values.sum() == 0

assert stage1c[RAW_FIELDS].equals(raw_snapshot)

print()
print(
    "PASS: Stage 1C safe field normalization succeeded."
)


STAGE 1C FINAL VALIDATION
Total records              : 177,638
Unique source IDs          : 177,638
Duplicate source IDs       : 0
Number of columns          : 24
Missing product names      : 0
Missing source categories  : 0
Missing normalized brands  : 6,411
Product-price parse errors : 0
Original-price parse errors: 0
Unknown stock values       : 0
Unknown offer values       : 0

PASS: Stage 1C safe field normalization succeeded.


## Stage 1C result

When the final validation passes:

- all **177,638** Stage 1B records are still present;
- all source record IDs are preserved;
- raw source values are unchanged;
- categories, product names and brands have conservative normalized counterparts;
- prices have numeric normalized counterparts;
- stock and offer flags have nullable Boolean counterparts;
- missing or placeholder brands remain missing rather than being guessed;
- specification parsing has not yet been attempted.

The next step is **Stage 1D: specification and pack-size parsing**, where `specification_raw` will be converted cautiously into fields such as `pack_count`, `unit_size`, `unit`, `total_quantity`, and `spec_parse_status`.
